In [1]:
import torch
import torchvision

In [2]:
torch.__version__, torchvision.__version__

('2.5.1+cu121', '0.20.1+cu121')

In [ ]:
!pip install -q git+https://github.com/google-deepmind/tapnet.git

In [ ]:
!pip install -q git+https://github.com/google-deepmind/recurrentgemma.git@main

In [ ]:
!pip install "numpy<2.1.0"

In [3]:
import tqdm

### TAPNext

In [5]:
import time
import numpy as np
from tapnet.tapnext.tapnext_torch import TAPNext
import torch.nn.functional as F

/home/student/anaconda3/envs/tjc-tap/lib/python3.10/site-packages/torch/_export/utils.py:394: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _register_pytree_node(


### Create the model and load checkpoint

In [6]:
model = TAPNext(image_size=(256, 256))
model.cuda()

TAPNext(
  (lin_proj): Conv2d(3, 768, kernel_size=(8, 8), stride=(8, 8))
  (blocks): ModuleList(
    (0-11): 12 x TRecViTBlock(
      (ssm_block): ResidualBlock(
        (temporal_pre_norm): RMSNorm()
        (recurrent_block): RecurrentBlock(
          (linear_y): Linear(in_features=768, out_features=768, bias=True)
          (linear_x): Linear(in_features=768, out_features=768, bias=True)
          (linear_out): Linear(in_features=768, out_features=768, bias=True)
          (conv_1d): CausalConv1D()
          (rg_lru): RGLRU(
            (input_gate): BlockDiagonalLinear()
            (a_gate): BlockDiagonalLinear()
          )
        )
        (channel_pre_norm): RMSNorm()
        (mlp_block): MLPBlock(
          (ffw_up): Einsum()
          (ffw_down): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (vit_block): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (

### Run inference

In [7]:
model.eval()
for p in model.parameters():
  p.requires_grad = False

In [8]:
NUM_QUERIES = 1024

In [9]:
video = torch.zeros((1, 1024, 256, 256, 3), dtype=torch.float32).cuda()
query_points = torch.zeros((1, NUM_QUERIES, 3), dtype=torch.float32).cuda()

In [10]:
DTYPE = torch.float16  # use fp16 or bf16

In [11]:
fwd = torch.compile(model.forward)
with torch.no_grad():
  with torch.amp.autocast('cuda', dtype=DTYPE, enabled=True):
    _, _, _, tracking_state = fwd(video=video[:, :1], query_points=query_points)
    c = 0
    for k in tqdm.tqdm(range(1, video.shape[1])):
      if k == 512:
        # we let the model to run for several GPU burn-in steps
        tt = time.time()
        c = 0
      _, _, _, tracking_state = fwd(
          video=video[:, k : k + 1], state=tracking_state
      )
      c += 1
    d = time.time() - tt
    print('FPS:', c / d, 'latency', 1000 * d / c, 'ms')

100%|██████████| 1023/1023 [00:58<00:00, 17.61it/s]

FPS: 88.9447427561773 latency 11.242935433983803 ms
